In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import os
import glob

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc/'
nuclei = pd.Index(np.loadtxt(f'{projdir}/txt/mc_nuclei.txt', dtype=str))


### get allc paths

In [ ]:
# indir = '/u/project/cluo/IGVF_Share/year2_pgp_dense/snmct/allc'
# allc_paths = glob.glob(f'{indir}/*.allc.tsv.gz')
# len(allc_paths)

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/allc/raw'
allc_paths = glob.glob(f'{indir}/*.CGN-Merge.allc.tsv.gz')
len(allc_paths)

In [ ]:
# mask = np.array([os.path.exists(x) for x in allc_paths])
# final_allc_paths = np.array(allc_paths)[mask]
# final_allc_paths.shape

In [ ]:
tmp_df = pd.DataFrame(allc_paths)
tmp_df.index = [x.split('/')[-1].split('.')[0] for x in tmp_df[0]]
# tmp_df = tmp_df[~tmp_df.index.duplicated()]
tmp_df.shape

In [ ]:
meta_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/meta/mC_meta.csv'
meta = pd.read_csv(meta_path, sep='\t', header=0, index_col=0)
meta = meta[meta.index.isin(nuclei)]
meta.shape

In [ ]:
meta.head()

In [ ]:
meta['allc_path'] = tmp_df.reindex(tmp_df.index)
final_meta = meta.copy().dropna()
final_meta.shape

### add rna_cluster_labels

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna'
rna_clusters = pd.read_csv(f'{indir}/csv/label_transfer/snmct_xgboost_v7.csv', sep='\t', header=0, index_col=0)
rna_clusters.shape

In [ ]:
rna_clusters.head()

In [ ]:
final_meta['rna_cluster'] = rna_clusters.reindex(final_meta.index)['pred_coarse']
final_meta['rna_cluster_donor'] = [f'{x}_{y}' for x,y in zip(final_meta['rna_cluster'], final_meta['donor'])]
cluster_meta = final_meta[~final_meta['rna_cluster'].isna()]
cluster_meta.shape

In [ ]:
cluster_meta['rna_cluster'].value_counts()

In [ ]:
cluster_meta['rna_cluster_donor'].value_counts().sort_index()

In [ ]:
cluster_meta.head()

In [ ]:
donors = sorted(cluster_meta['donor'].unique())
s = 'Start'
tmp_meta = cluster_meta[cluster_meta['rna_cluster'] == s]

for d in donors:
    donor_meta = tmp_meta[tmp_meta['donor'] == d]
    print(donor_meta['time'].value_counts())

In [ ]:
donor_meta

### write out

In [ ]:
cluster_meta

In [ ]:
design = 'donor_time'

clusts = cluster_meta[design].unique()
outdir = f'{projdir}/txt/allc_table/{design}'

for clust in clusts:
#     print(clust)
    mask = cluster_meta[design] == clust
    tmp_allc_paths = cluster_meta[mask][['allc_path']]
    
    ### double checking that it exists
    tmp_allc_paths.to_csv(f'{outdir}/{clust}.txt', sep='\t', header=False, index=False)
    
pd.DataFrame(clusts).to_csv(f'{projdir}/txt/allc_table/{design}.txt',
                            sep='\t', header=None, index=None)

In [ ]:
# clusts = final_meta['donor_time'].unique()
# outdir = f'{projdir}/csv/allc_table/donor_time/'

# for clust in clusts:
#     print(clust)
#     mask = final_meta['donor_time'] == clust
#     tmp_allc_paths = final_meta[mask][['allc_path']]
    
#     ### double checking that it exists
#     tmp_allc_paths.to_csv(f'{outdir}/{clust}.csv', sep='\t', header=False, index=False)
    
# pd.DataFrame(clusts).to_csv(f'{projdir}/txt/allc_table/donor_time.txt', sep='\t', header=None, index=None)

In [ ]:
design = 'rna_cluster_donor'

clusts = cluster_meta[design].unique()
outdir = f'{projdir}/txt/allc_table/{design}'

for clust in clusts:
#     print(clust)
    mask = cluster_meta[design] == clust
    tmp_allc_paths = cluster_meta[mask][['allc_path']]
    
    ### double checking that it exists
    tmp_allc_paths.to_csv(f'{outdir}/{clust}.txt', sep='\t', header=False, index=False)
    
pd.DataFrame(clusts).to_csv(f'{projdir}/txt/allc_table/{design}.txt',
                            sep='\t', header=None, index=None)

In [ ]:
design = 'rna_cluster'

clusts = cluster_meta[design].unique()
outdir = f'{projdir}/txt/allc_table/{design}'

for clust in clusts:
#     print(clust)
    mask = cluster_meta[design] == clust
    tmp_allc_paths = cluster_meta[mask][['allc_path']]
    
    ### double checking that it exists
    tmp_allc_paths.to_csv(f'{outdir}/{clust}.txt', sep='\t', header=False, index=False)
    
pd.DataFrame(clusts).to_csv(f'{projdir}/txt/allc_table/{design}.txt',
                            sep='\t', header=None, index=None)

In [ ]:
cluster_meta['time_num'] = [x.split('D')[1] for x in cluster_meta['time']]
cluster_meta['rna_cluster_donor_time'] = [f'{x}_{y}' for x,y in zip(cluster_meta['rna_cluster_donor'], cluster_meta['time_num'])]

In [ ]:
dist = cluster_meta['rna_cluster_donor_time'].value_counts()
dist

In [ ]:
n_cells = 50
mask = (dist > n_cells)
final_cdt = dist[mask].index
final_cdt.shape

In [ ]:
final_cdt[:5]

In [ ]:
mask = cluster_meta['rna_cluster_donor_time'].isin(final_cdt)
final_meta = cluster_meta[mask]
final_meta.shape

In [ ]:
design = 'rna_cluster_donor_time'

clusts = final_meta[design].unique()
outdir = f'{projdir}/txt/allc_table/{design}'

for clust in clusts:
#     print(clust)
    mask = final_meta[design] == clust
    tmp_allc_paths = final_meta[mask][['allc_path']]
    
    ### double checking that it exists
    tmp_allc_paths.to_csv(f'{outdir}/{clust}.txt', sep='\t', header=False, index=False)
    
pd.DataFrame(clusts).to_csv(f'{projdir}/txt/allc_table/{design}.txt',
                            sep='\t', header=None, index=None)

### rna leiden

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna'
rna_leiden = pd.read_csv(f'{indir}/csv/clusters/leiden_filtered.csv', sep='\t', index_col=0)
rna_leiden.shape

In [ ]:
rna_leiden.head()

In [ ]:
# design = 'rna_leiden'

# clusts = final_meta[design].unique()
# outdir = f'{projdir}/csv/allc_table/{design}'

# for clust in clusts:
# #     print(clust)
#     mask = final_meta[design] == clust
#     tmp_allc_paths = final_meta[mask][['allc_path']]
#     tmp_allc_paths.to_csv(f'{outdir}/{clust}.csv', sep='\t', header=False, index=False)
    
# pd.DataFrame(clusts).to_csv(f'{projdir}/txt/allc_table/{design}.txt',
#                             sep='\t', header=None, index=None)

### mCG per-donor leiden

In [ ]:
clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/integrate_m3c/leiden_clusters_renamed.csv'
leiden = pd.read_csv(clusters_path, sep='\t', index_col=0)
leiden.shape

In [ ]:
leiden['renamed_leiden'] = leiden['renamed_leiden'].replace('Inter_2', 'Fibro_15')

In [ ]:
leiden['donor_leiden'] = [f'{x}_{y}' for x,y in zip(leiden['line'], leiden['renamed_leiden'])]
leiden['donor_leiden'].head()

In [ ]:
leiden_meta = final_meta.copy()
leiden_meta['donor_leiden'] = leiden.reindex(leiden_meta.index)['donor_leiden']
leiden_meta = leiden_meta[~leiden_meta['allc_path'].isna()]
leiden_meta.shape

In [ ]:
leiden_meta['allc_path'].isna().sum()

In [ ]:
leiden_meta.head()

In [ ]:
indir = '/u/home/t/terencew/project-cluo/igvf/2023_YR2/snmCT/mc/allc/raw'
mask = [os.path.exists(f'{indir}/{x}.CGN-Merge.allc.tsv.gz.tbi') for x in leiden_meta.index]
leiden_meta.shape, sum(mask)

In [ ]:
final_leiden_meta = leiden_meta[mask]
final_leiden_meta.shape

In [ ]:
# tmp = leiden[leiden['line'] == 'C39']
# tmp['renamed_leiden'].value_counts()

In [ ]:
final_leiden_meta.shape

In [ ]:
design = 'donor_leiden'

clusts = final_leiden_meta[design].unique()
outdir = f'{projdir}/txt/allc_table/{design}'

for clust in clusts:
#     print(clust)
    mask = final_leiden_meta[design] == clust
    tmp_allc_paths = final_leiden_meta[mask][['allc_path']]
    tmp_allc_paths.to_csv(f'{outdir}/{clust}.txt', sep='\t', header=False, index=False)
    
pd.DataFrame(clusts).to_csv(f'{projdir}/txt/allc_table/{design}.txt',
                            sep='\t', header=None, index=None)

In [ ]:
# tmp_meta = final_leiden_meta[final_leiden_meta['rna_cluster'] == 'Start'].copy()
tmp_meta = final_leiden_meta[final_leiden_meta['time'] == 'D0'].copy()
tmp_meta['donor_leiden_fibro'] = tmp_meta['donor_leiden'].copy()
tmp_meta.shape

In [ ]:
design = 'donor_leiden_fibro'

clusts = tmp_meta[design].unique()
outdir = f'{projdir}/txt/allc_table/{design}'

for clust in clusts:
#     print(clust)
    mask = tmp_meta[design] == clust
    tmp_allc_paths = tmp_meta[mask][['allc_path']]
    tmp_allc_paths.to_csv(f'{outdir}/{clust}.txt', sep='\t', header=False, index=False)
    
pd.DataFrame(clusts).to_csv(f'{projdir}/txt/allc_table/{design}.txt',
                            sep='\t', header=None, index=None)

### subcluster for justin

In [ ]:
rev_class_path = '/u/project/cluo/Shared_Datasets/IGVF/202208_Pilot/new_multiome/YS3v7_Data/CL1.metadata.with.ys3_v7_revclass.txt'
rev_class = pd.read_csv(rev_class_path, sep='\t', header=0)
rev_class.shape

In [ ]:
rev_class.head()

In [ ]:
rev_class['RevClass2'].value_counts()

In [ ]:
rev_class['rev_donor'] = [f'{x}_{y}' for x,y in zip(rev_class['RevClass2'], rev_class['Donor'])]

In [ ]:
final_meta.shape

In [ ]:
final_meta['rev_class'] = rev_class.reindex(final_meta.index)['RevClass2']
final_meta['rev_donor'] = rev_class.reindex(final_meta.index)['rev_donor']

In [ ]:
design = 'rev_class'

clusts = final_meta[design].unique()
outdir = f'{projdir}/txt/allc_table/{design}'

for clust in clusts:
#     print(clust)
    mask = final_meta[design] == clust
    tmp_allc_paths = final_meta[mask][['allc_path']]
    tmp_allc_paths.to_csv(f'{outdir}/{clust}.txt', sep='\t', header=False, index=False)
    
pd.DataFrame(clusts).to_csv(f'{projdir}/txt/allc_table/{design}.txt',
                            sep='\t', header=None, index=None)

In [ ]:
design = 'rev_donor'

clusts = final_meta[design].unique()
outdir = f'{projdir}/txt/allc_table/{design}'

for clust in clusts:
#     print(clust)
    mask = final_meta[design] == clust
    tmp_allc_paths = final_meta[mask][['allc_path']]
    tmp_allc_paths.to_csv(f'{outdir}/{clust}.txt', sep='\t', header=False, index=False)
    
pd.DataFrame(clusts).to_csv(f'{projdir}/txt/allc_table/{design}.txt',
                            sep='\t', header=None, index=None)

### ips subcluster

In [ ]:
ips_clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/label_transfer/snm3c_ips_subtype.csv'
ips_clusters = pd.read_csv(ips_clusters_path, sep='\t', index_col=0)
ips_clusters.shape

In [ ]:
ips_clusters.head()

In [ ]:
final_meta['ips_cluster'] = ips_clusters.reindex(final_meta.index)['ips_subtype']

In [ ]:
ips_meta = final_meta[~final_meta['ips_cluster'].isna()].copy()
ips_meta['ips_cluster_donor'] = [f'{x}_{y}' for x,y in zip(ips_meta['ips_cluster'], ips_meta['donor'])]
ips_meta.shape

In [ ]:
design = 'ips_cluster'

clusts = ips_meta[design].unique()
outdir = f'{projdir}/txt/allc_table/{design}'

for clust in clusts:
#     print(clust)
    mask = ips_meta[design] == clust
    tmp_allc_paths = ips_meta[mask][['allc_path']]
    tmp_allc_paths.to_csv(f'{outdir}/{clust}.txt', sep='\t', header=False, index=False)
    
pd.DataFrame(clusts).to_csv(f'{projdir}/txt/allc_table/{design}.txt',
                            sep='\t', header=None, index=None)

In [ ]:
design = 'ips_cluster_donor'

clusts = ips_meta[design].unique()
outdir = f'{projdir}/txt/allc_table/{design}'

for clust in clusts:
#     print(clust)
    mask = ips_meta[design] == clust
    tmp_allc_paths = ips_meta[mask][['allc_path']]
    tmp_allc_paths.to_csv(f'{outdir}/{clust}.txt', sep='\t', header=False, index=False)
    
pd.DataFrame(clusts).to_csv(f'{projdir}/txt/allc_table/{design}.txt',
                            sep='\t', header=None, index=None)

In [ ]:
!date